# 12 K-Means 聚类

K-Means 是无监督学习模型，用来把样本分成 K 个簇。它没有标签，目标是让每个点离自己的簇中心尽量近。


## 0. 学习目标和阅读地图

K-Means 是无监督学习的经典入口。你需要掌握：

1. 聚类没有真实标签时，模型到底在优化什么。
2. inertia 和 silhouette 分别怎么看。
3. K 值和初始化为什么重要。
4. K-Means 适合什么形状的簇。


## 1. 数学逻辑

K-Means 最小化簇内平方距离：

$$\min_{C,\mu}\sum_{k=1}^{K}\sum_{x_i\in C_k}||x_i-\mu_k||^2$$

算法循环两步：

1. 分配：每个点归到最近的中心。
2. 更新：每个中心移动到自己簇内点的均值。


## 1.1 推导拆开看：为什么均值是最佳中心

当簇分配固定时，第 k 个簇的目标是：

$$\min_{\mu_k}\sum_{x_i\in C_k}||x_i-\mu_k||^2$$

对 `mu_k` 求导并令其为 0，可以得到：

$$\mu_k=\frac{1}{|C_k|}\sum_{x_i\in C_k}x_i$$

所以 K-Means 的更新步骤就是把中心移到簇内点的均值。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager


def setup_chinese_font():
    candidates = [
        'PingFang SC',
        'Heiti SC',
        'Songti SC',
        'Arial Unicode MS',
        'Noto Sans CJK SC',
        'Noto Sans SC',
        'SimHei',
        'Microsoft YaHei',
        'WenQuanYi Micro Hei',
    ]
    available = {font.name for font in font_manager.fontManager.ttflist}
    for font in candidates:
        if font in available:
            existing = [name for name in plt.rcParams['font.sans-serif'] if name != font]
            plt.rcParams['font.family'] = 'sans-serif'
            plt.rcParams['font.sans-serif'] = [font] + existing
            break
    else:
        print('Warning: no Chinese font found. Install Noto Sans CJK SC or SimHei if Chinese text is missing in plots.')
    plt.rcParams['axes.unicode_minus'] = False


setup_chinese_font()

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

np.random.seed(42)
X, _ = make_blobs(n_samples=260, centers=3, cluster_std=0.7, random_state=42)
plt.scatter(X[:,0], X[:,1], s=24)
plt.title('没有标签的点云')
plt.show()


## 1.2 K-Means 的两个循环

K-Means 在两个动作之间反复切换：

- assignment：每个样本找最近中心。
- update：每个中心变成簇内样本均值。

每轮都会让 inertia 不增加，但不保证找到全局最优，所以需要多次初始化。


In [ ]:
# 从零实现 K-Means
K = 3
rng = np.random.default_rng(42)
centers = X[rng.choice(len(X), size=K, replace=False)]

for step in range(15):
    distances = np.sqrt(((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2))
    labels = distances.argmin(axis=1)
    new_centers = np.array([X[labels == k].mean(axis=0) for k in range(K)])
    shift = np.sqrt(((new_centers - centers) ** 2).sum())
    centers = new_centers
    print(f'step {step+1:2d} | center shift={shift:.4f}')
    if shift < 1e-4:
        break

plt.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', s=24)
plt.scatter(centers[:,0], centers[:,1], c='red', marker='x', s=120)
plt.title('从零 K-Means 结果')
plt.show()


## 1.3 从零实现代码怎么读

`distances` 的形状是 `(样本数, K)`，表示每个样本到每个中心的距离。`argmin(axis=1)` 选出每个样本最近的中心。

`center shift` 是中心移动距离。如果中心几乎不动，说明算法收敛了。


In [ ]:
model = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = model.fit_predict(X)
print('inertia:', round(model.inertia_, 3))
print('silhouette:', round(silhouette_score(X, labels), 3))

plt.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', s=24)
plt.scatter(model.cluster_centers_[:,0], model.cluster_centers_[:,1], c='red', marker='x', s=120)
plt.title('sklearn KMeans')
plt.show()


In [ ]:
# 诊断：elbow 和 silhouette 帮助选择 K
ks = range(2, 8)
inertias, silhouettes = [], []
for k in ks:
    m_kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    lab = m_kmeans.fit_predict(X)
    inertias.append(m_kmeans.inertia_)
    silhouettes.append(silhouette_score(X, lab))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(list(ks), inertias, marker='o')
plt.title('Elbow: K 与 inertia')
plt.xlabel('K')
plt.ylabel('inertia')

plt.subplot(1, 2, 2)
plt.plot(list(ks), silhouettes, marker='o')
plt.title('K 与 silhouette')
plt.xlabel('K')
plt.ylabel('silhouette')
plt.tight_layout()
plt.show()


## 2.1 如何诊断 K-Means

inertia 一定会随 K 增大而下降，因为更多中心总能让点离中心更近。所以不能只看 inertia 最小。

silhouette 会同时考虑簇内紧密和簇间分离，通常更适合比较不同 K，但也不是绝对真理。


## 2. 常见误区

- K-Means 需要预先指定 K。
- 它偏好球形、大小接近的簇；对月牙形、密度不均数据效果差。
- 初始化会影响结果，所以通常多次初始化。

## 3. 小实验

- 把 `K` 改成 2 或 4。
- 增大 `cluster_std`，观察簇重叠。
- 换成 `make_moons`，看 K-Means 的局限。


## 5. 复习清单

- K-Means 优化簇内平方距离。
- 必须指定 K。
- 适合球形、大小接近的簇。
- 初始化会影响结果，通常使用 `n_init > 1`。
